# New

In [1]:
import pdfplumber
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import os
import warnings
warnings.filterwarnings("ignore")

# -------------------------------
# STEP 1: Load BERT
# -------------------------------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [2]:
# -------------------------------
# STEP 2: Extract text from PDF
# -------------------------------
def extract_text_from_pdf(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [3]:
# -------------------------------
# STEP 3: Skill Extraction
# -------------------------------
SKILLS_DB = [
    "python", "machine learning", "deep learning",
    "nlp", "sql", "tensorflow", "pytorch",
    "excel", "data analysis", "java", "c++"
]

def extract_skills(text):
    text = text.lower()
    found = [skill for skill in SKILLS_DB if skill in text]
    return list(set(found))

In [4]:
# -------------------------------
# STEP 4: BERT Embedding
# -------------------------------
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state[:, 0, :]  # CLS token

In [5]:

# -------------------------------
# STEP 5: Similarity
# -------------------------------
def compute_similarity(resume_text, jd_text):
    emb1 = get_embedding(resume_text)
    emb2 = get_embedding(jd_text)
    score = cosine_similarity(emb1.numpy(),emb2.numpy())[0][0]
    return score

In [6]:
# -------------------------------
# STEP 6: Process Multiple PDFs
# -------------------------------
def process_resumes(pdf_folder, job_description):
    results = []

    for file in os.listdir(pdf_folder):
        if file.endswith(".pdf"):
            path = os.path.join(pdf_folder, file)

            print(f"Processing: {file}")

            text = extract_text_from_pdf(path)
            skills = extract_skills(text)
            score = compute_similarity(text, job_description)

            results.append({
                "name": file,
                "text":text,
                "skills": skills,
                "score": score
            })

    # Rank resumes
    results = sorted(results, key=lambda x: x["score"], reverse=True)

    return results



In [7]:

# -------------------------------
# STEP 7: Run Example
# -------------------------------
if __name__ == "__main__":

    # Folder with 5 PDF resumes
    pdf_folder = "D:/Python Code/Resume_IRFAN/"   # <-- put your 5 PDFs here

    job_description = """
    Looking for a Python developer with experience in
    machine learning, NLP, and data analysis.
    """

    results = process_resumes(pdf_folder, job_description)

    print("\n===== FINAL RANKING =====\n")

    for i, res in enumerate(results, 1):
        print(f"Rank {i}")
        print(f"Resume: {res['name']}")
        print(f"Score: {res['score']:.3f}")
        print(f"Skills: {res['skills']}")
        #print(f"text: {res['text']}")
        print("-" * 40)

Processing: Irfan updated by today V3 -- Final.pdf
Processing: Irfan updated by today V4.pdf

===== FINAL RANKING =====

Rank 1
Resume: Irfan updated by today V4.pdf
Score: 0.715
Skills: ['sql', 'python', 'excel', 'deep learning', 'machine learning', 'nlp']
----------------------------------------
Rank 2
Resume: Irfan updated by today V3 -- Final.pdf
Score: 0.675
Skills: ['sql', 'python', 'excel', 'deep learning', 'machine learning', 'nlp']
----------------------------------------


# Named Entity Recognition (NER) with a BERT Model

In [9]:
import warnings
warnings.filterwarnings('ignore')
from transformers import pipeline
from transformers import logging
logging.set_verbosity_error()


In [10]:
# Initialize the NER pipeline with a pre-trained BERT model ( Base Model )
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

# Define sample text
text = "Apple Inc. plans to open a new store in San Francisco by January 2024. Tim Cook announced the news."

# Perform NER
entities = ner_pipeline(text)

# Print results
for entity in entities:
    print(f"Entity: {entity['word']} | Label: {entity['entity_group']} | Score: {entity['score']:.4f}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Entity: Apple Inc | Label: ORG | Score: 0.9995
Entity: San Francisco | Label: LOC | Score: 0.9994
Entity: Tim Cook | Label: PER | Score: 0.9998


In [11]:
from transformers import pipeline #(Large Model)

# Initialize the NER pipeline
ner_pipeline = pipeline("ner", 
                        model="dbmdz/bert-large-cased-finetuned-conll03-english",
                        aggregation_strategy="simple")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [12]:
# Text example
text = "Apple CEO Tim Cook announced new iPhone models in California yesterday."

# Perform NER
entities = ner_pipeline(text)

# Print the results
for entity in entities:
    print(f"Entity: {entity['word']}")
    print(f"Type: {entity['entity_group']}")
    print(f"Confidence: {entity['score']:.4f}")
    print("-" * 30)

Entity: Apple
Type: ORG
Confidence: 0.9975
------------------------------
Entity: Tim Cook
Type: PER
Confidence: 0.9996
------------------------------
Entity: iPhone
Type: MISC
Confidence: 0.9932
------------------------------
Entity: California
Type: LOC
Confidence: 0.9997
------------------------------
